In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find MedGemma-27b-text-it project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"

# Full path to the model snapshot
model_path = "/orcd/compute/mghassem/001/gobi1/huggingface/hub/models--google--medgemma-27b-text-it/snapshots/5b667cf2ddcf064085bc90952edb35a0edbfb79c"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█████████████████████████████████████████| 11/11 [00:07<00:00,  1.44it/s]
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to understand, generate, and interact with human language.**

Think of them as incredibly sophisticated pattern-matching machines. They are trained on massive amounts of text data (like books, articles, websites) and learn the statistical relationships between words and concepts.

**Key characteristics:**

*   **"Large":** They have billions (or even trillions) of parameters, which are essentially the variables the model adjusts during training to learn patterns.
*   **"Language":** Their primary function is processing and generating human language.
*   **Capabilities:** They can perform tasks like:
    *   Answering questions
    *   Writing essays, code, or creative content
    *   Translating languages
    *   Summarizing text
    *   Holding conversations (like chatbots)

**In essence, LLMs are powerful tools that can mimic human-lik

In [2]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "MedGemma27B_predictions_on_Trainee_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["New_Sentences"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        prompt = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Use chat template format (like your working example)
        messages = [
            {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate prediction
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated part (not the input)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode the response
        raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract the answer letter
        extracted_answer = extract_answer_letter(raw_response)
        
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "MedGemma27B_predictions_on_Trainee.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Origin', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'ID_corr', 'sentence_number_corr', 'answer_corr', 'data_source_corr', 'REMOVED_Sentences', 'sentence_number_df3', 'step1_excerpts', 'question_options', 'Filtered_Sentences', 'New_Sentences']
Found 1020 already processed rows. Resuming...
Processing 1300 rows...
Skipping row 1/1300 (already processed)...
Skipping row 2/1300 (already processed)...
Skipping row 3/1300 (already processed)...
Skipping row 4/1300 (already processed)...
Skipping row 5/1300 (already processed)...
Skipping row 6/1300 (already processed)...
Skipping row 7/1300 (already processed)...
Skipping row 8/1300 (already processed)...
Skipping row 9/1300 (already processed)...
Skipping row 10/1300 (already processed)...
Skipping row 11/1300 (already processed)...
Skipping row 12/1300 (already processed)...
Skipping row 13/1300 (already processed)...
Skipping row 

✅ Processed Merge Q1021: Answer = J
Processing row 1022/1300...
✅ Processed Merge Q1022: Answer = B
Processing row 1023/1300...
⚠️ Could not extract answer from response for row 1023:
Response: <unused94>thought
The user wants me to analyze a clinical case and choose the most appropriate next ...
✅ Processed Merge Q1023: Answer = None
Processing row 1024/1300...
⚠️ Could not extract answer from response for row 1024:
Response: <unused94>thought
The user wants me to identify the mechanism of action of desmopressin based on the...
✅ Processed Merge Q1024: Answer = None
Processing row 1025/1300...
⚠️ Could not extract answer from response for row 1025:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the best nex...
✅ Processed Merge Q1025: Answer = None
Processing row 1026/1300...
✅ Processed Merge Q1026: Answer = C
Processing row 1027/1300...
⚠️ Could not extract answer from response for row 1027:
Response: <unused94>thought
The user want

✅ Processed Merge Q1071: Answer = A
Processing row 1072/1300...
✅ Processed Merge Q1072: Answer = F
Processing row 1073/1300...
⚠️ Could not extract answer from response for row 1073:
Response: <unused94>thought
The user wants me to analyze a patient case and choose the most appropriate next s...
✅ Processed Merge Q1073: Answer = None
Processing row 1074/1300...
✅ Processed Merge Q1074: Answer = B
Processing row 1075/1300...
✅ Processed Merge Q1075: Answer = D
Processing row 1076/1300...
✅ Processed Merge Q1076: Answer = B
Processing row 1077/1300...
⚠️ Could not extract answer from response for row 1077:
Response: <unused94>thought
The patient is a 43-year-old man with Crohn disease on ustekinumab presenting with...
✅ Processed Merge Q1077: Answer = None
Processing row 1078/1300...
⚠️ Could not extract answer from response for row 1078:
Response: <unused94>thought
The user wants me to identify the most likely Chapman reflex point based on the pr...
✅ Processed Merge Q1078: Answer = No

⚠️ Could not extract answer from response for row 1132:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the best nex...
✅ Processed Merge Q1132: Answer = None
Processing row 1133/1300...
✅ Processed Merge Q1133: Answer = B
Processing row 1134/1300...
✅ Processed Merge Q1134: Answer = B
Processing row 1135/1300...
⚠️ Could not extract answer from response for row 1135:
Response: <unused94>thought
The user wants me to identify the most concerning risk factor for bladder cancer i...
✅ Processed Merge Q1135: Answer = None
Processing row 1136/1300...
⚠️ Could not extract answer from response for row 1136:
Response: <unused94>thought
The user wants me to identify the most appropriate next step in evaluating a 2.5-y...
✅ Processed Merge Q1136: Answer = None
Processing row 1137/1300...
✅ Processed Merge Q1137: Answer = C
Processing row 1138/1300...
✅ Processed Merge Q1138: Answer = D
Processing row 1139/1300...
⚠️ Could not extract answer from 

⚠️ Could not extract answer from response for row 1187:
Response: <unused94>thought
The user wants me to identify the best parameter from the provided options that su...
✅ Processed Merge Q1187: Answer = None
Processing row 1188/1300...
⚠️ Could not extract answer from response for row 1188:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the most app...
✅ Processed Merge Q1188: Answer = None
Processing row 1189/1300...
⚠️ Could not extract answer from response for row 1189:
Response: <unused94>thought
The user wants me to analyze a patient case and choose the most appropriate next s...
✅ Processed Merge Q1189: Answer = None
Processing row 1190/1300...
✅ Processed Merge Q1190: Answer = D
Saved progress to CSV after 1190 items
Processing row 1191/1300...
✅ Processed Merge Q1191: Answer = A
Processing row 1192/1300...
✅ Processed Merge Q1192: Answer = B
Processing row 1193/1300...
✅ Processed Merge Q1193: Answer = C
Processing row 1194/13

⚠️ Could not extract answer from response for row 1247:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the best nex...
✅ Processed Merge Q1247: Answer = None
Processing row 1248/1300...
✅ Processed Merge Q1248: Answer = B
Processing row 1249/1300...
⚠️ Could not extract answer from response for row 1249:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the most app...
✅ Processed Merge Q1249: Answer = None
Processing row 1250/1300...
✅ Processed Merge Q1250: Answer = G
Saved progress to CSV after 1250 items
Processing row 1251/1300...
✅ Processed Merge Q1251: Answer = A
Processing row 1252/1300...
✅ Processed Merge Q1252: Answer = A
Processing row 1253/1300...
✅ Processed Merge Q1253: Answer = C
Processing row 1254/1300...
⚠️ Could not extract answer from response for row 1254:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the best nex.

In [6]:
#### import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "MedGemma27B_predictions_on_Trainee.csv")
df = pd.read_csv(paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate None/Empty cells in Extracted_Answer
total_none_empty = output_df['Extracted_Answer'].isna().sum() + (output_df['Extracted_Answer'] == '').sum()
total_cells = len(output_df)
none_empty_percentage = (total_none_empty / total_cells) * 100

print("=" * 60)
print("NONE/EMPTY CELL ANALYSIS")
print("=" * 60)
print(f"Total None/Empty cells in 'Extracted_Answer': {total_none_empty}")
print(f"Total cells: {total_cells}")
print(f"Percentage None/Empty: {none_empty_percentage:.2f}%")
print("=" * 60)
print()

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_corr' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_corr' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_corr'] = df['data_source_corr']
else:
    print("Warning: 'data_source_corr' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_corr' in analysis_df.columns:
    # Calculate None/Empty cells by data_source
    none_empty_by_source = analysis_df.groupby('data_source_corr').apply(
        lambda x: (x['Extracted_Answer'].isna().sum() + (x['Extracted_Answer'] == '').sum())
    ).reset_index(name='None_Empty_Count')
    
    source_totals = analysis_df.groupby('data_source_corr').size().reset_index(name='Total_Count')
    none_empty_summary = none_empty_by_source.merge(source_totals, on='data_source_corr')
    none_empty_summary['None_Empty_Percentage'] = (none_empty_summary['None_Empty_Count'] / none_empty_summary['Total_Count']) * 100
    
    print("NONE/EMPTY CELLS BY DATA SOURCE")
    print("=" * 60)
    print(none_empty_summary.to_string(index=False))
    print("=" * 60)
    print()
    
    category_stats = analysis_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

NONE/EMPTY CELL ANALYSIS
Total None/Empty cells in 'Extracted_Answer': 542
Total cells: 1300
Percentage None/Empty: 41.69%

OVERALL ACCURACY ANALYSIS
Accuracy: 0.3777 (37.77%)
Standard Deviation: 0.4850
95% Confidence Interval: [0.3513, 0.4041]
95% CI (percentage): [35.13%, 40.41%]
Sample Size: 1300

NONE/EMPTY CELLS BY DATA SOURCE
data_source_corr  None_Empty_Count  Total_Count  None_Empty_Percentage
            jama               239          582              41.065292
      medbullets                62          207              29.951691
        medxpert               174          318              54.716981
            mmlu                67          193              34.715026

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.383162 0.486575 0.020169     0.343548     0.422775        38.316151  48.657534      34.354809      42.277

/tmp/ipykernel_1367632/2190579698.py:62: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  none_empty_by_source = analysis_df.groupby('data_source_corr').apply(


In [9]:
import pandas as pd
import numpy as np
from scipy import stats

output_df = pd.read_csv(paths.PREDICTIONS / "MedGemma27B_predictions_on_Trainee.csv")
output_df

,QA_ID,Origin,data_source,Raw_Response,Physician_Removed_Answer,Original,answer_corr
0,Merge Q1,ID0002,jama,<unused94>thought\nThe user wants me to act as...,NaN,D,D
1,Merge Q2,ID0003,medxpert,<unused94>thought\nThe user wants me to identi...,NaN,H,G
2,Merge Q3,ID0007,medbullets,<unused94>thought\nThe user wants me to identi...,NaN,B,B
3,Merge Q4,ID0009,jama,Answer: D,D,D,D
4,Merge Q5,ID0010,medxpert,Answer: H,H,H,H
...,...,...,...,...,...,...,...
1295,Merge Q1296,ID1995,mmlu,<unused94>thought\nThe user wants me to identi...,NaN,D,D
1296,Merge Q1297,ID1996,mmlu,<unused94>thought\nThe user wants me to identi...,NaN,D,D
1297,Merge Q1298,ID1997,medxpert,Answer: C,C,C,B
1298,Merge Q1299,ID1998,medbullets,<unused94>thought\nThe user wants me to identi...,NaN,B,A


In [10]:
# Calculate spurious rate function
def calculate_spurious_rate(df, original_col, comparison_col, correct_answer_col):
    """
    Calculate the percentage of questions that were:
    1. Answered CORRECTLY in the original_col (Original == answer_corr)
    2. Answered INCORRECTLY in the comparison_col (comparison_col != answer_corr)
    
    Spurious rate = (count of [Original correct AND comparison incorrect]) / (count of [Original correct])
    """
    # Step 1: Filter rows where Original is correct and has valid data
    original_correct_mask = (df[original_col].notna()) & (df[original_col] == df[correct_answer_col])
    original_correct = df[original_correct_mask].copy()
    
    total_original_correct = len(original_correct)
    
    if total_original_correct == 0:
        return 0.0, 0, 0
    
    # Step 2: Among those where Original was correct, find where comparison is incorrect
    comparison_incorrect_mask = (
        original_correct[comparison_col].notna() & 
        (original_correct[comparison_col] != original_correct[correct_answer_col])
    )
    
    spurious_count = comparison_incorrect_mask.sum()
    spurious_rate = (spurious_count / total_original_correct) * 100
    
    return spurious_rate, spurious_count, total_original_correct

# Overall spurious rate for Physician_Removed_Answer
print("="*80)
print("PHYSICIAN REMOVED - OVERALL SPURIOUS RATE")
print("="*80)
print(f"{'Model':<30} {'Spurious Rate (%)':<20} {'Count':<15} {'Total Correct in Original'}")
print("-"*80)

overall_rate, overall_count, overall_total = calculate_spurious_rate(
    output_df, 
    'Original',
    'Physician_Removed_Answer', 
    'answer_corr'
)

print(f"{'Physician_Removed_Answer':<30} {overall_rate:>18.2f}% {overall_count:>14} / {overall_total}")
print("\n")

# Breakdown by data source
print("="*80)
print("PHYSICIAN REMOVED - SPURIOUS RATES BY DATA SOURCE")
print("="*80)

# Check which data_source column exists in output_df
data_source_col = None
for col in ['data_source', 'data_source_corr', 'data_source_df3', 'data_source_corr_trainee']:
    if col in output_df.columns:
        data_source_col = col
        print(f"Using column: {data_source_col}")
        break

if data_source_col is None:
    print("ERROR: No data_source column found!")
    print(f"Available columns: {output_df.columns.tolist()}")
else:
    data_sources = output_df[data_source_col].dropna().unique()
    breakdown_results = {}
    
    print(f"\n{'Data Source':<20} {'Spurious Rate (%)':<20} {'Count':<15} {'Total Correct in Original'}")
    print("-"*80)
    
    for source in sorted(data_sources):
        source_df = output_df[output_df[data_source_col] == source]
        
        rate, count, total = calculate_spurious_rate(
            source_df,
            'Original',
            'Physician_Removed_Answer',
            'answer_corr'
        )
        
        breakdown_results[source] = {
            'rate': rate,
            'count': count,
            'total': total
        }
        
        print(f"{source:<20} {rate:>18.2f}% {count:>14} / {total}")
    
    # Create summary DataFrame
    summary_data = [{
        'Model': 'Physician_Removed_Answer',
        'Data Source': 'Overall',
        'Spurious Rate (%)': overall_rate,
        'Spurious Count': overall_count,
        'Total Original Correct': overall_total
    }]
    
    for source, stats in breakdown_results.items():
        summary_data.append({
            'Model': 'Physician_Removed_Answer',
            'Data Source': source,
            'Spurious Rate (%)': stats['rate'],
            'Spurious Count': stats['count'],
            'Total Original Correct': stats['total']
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\n")
    print("="*80)
    print("SUMMARY TABLE - PHYSICIAN REMOVED")
    print("="*80)
    print(summary_df.to_string(index=False))
    
    # Save results
    summary_df.to_csv(paths.TABLES / "physician_removed_spurious_rate_analysis.csv", index=False)
    
    print("\n\nResults saved to paths.TABLES / "physician_removed_spurious_rate_analysis.csv"")

# Additional diagnostic info
print("\n")
print("="*80)
print("DIAGNOSTIC INFO - PHYSICIAN REMOVED")
print("="*80)
print(f"Total rows in dataset: {len(output_df)}")
print(f"Rows with non-null 'Original': {output_df['Original'].notna().sum()}")
print(f"Rows with non-null 'Physician_Removed_Answer': {output_df['Physician_Removed_Answer'].notna().sum()}")
print(f"Rows where 'Original' is correct: {(output_df['Original'] == output_df['answer_corr']).sum()}")
print(f"Rows where 'Physician_Removed_Answer' is correct: {(output_df['Physician_Removed_Answer'] == output_df['answer_corr']).sum()}")

# Show data source distribution if available
if data_source_col:
    print("\n")
    print("Data source distribution:")
    print(output_df[data_source_col].value_counts())

# Additional analysis: Show examples of spurious errors
print("\n")
print("="*80)
print("EXAMPLES OF SPURIOUS ERRORS (First 10)")
print("="*80)
spurious_mask = (
    (output_df['Original'].notna()) & 
    (output_df['Original'] == output_df['answer_corr']) &
    (output_df['Physician_Removed_Answer'].notna()) &
    (output_df['Physician_Removed_Answer'] != output_df['answer_corr'])
)

spurious_examples = output_df[spurious_mask][['Original', 'Physician_Removed_Answer', 'answer_corr', data_source_col if data_source_col else 'Origin']].head(10)
print(spurious_examples.to_string(index=False))

PHYSICIAN REMOVED - OVERALL SPURIOUS RATE
Model                          Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
Physician_Removed_Answer                    10.66%             74 / 694


PHYSICIAN REMOVED - SPURIOUS RATES BY DATA SOURCE
Using column: data_source

Data Source          Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
jama                              14.53%             52 / 358
medbullets                         9.09%             11 / 121
medxpert                          12.31%              8 / 65
mmlu                               2.00%              3 / 150


SUMMARY TABLE - PHYSICIAN REMOVED
                   Model Data Source  Spurious Rate (%)  Spurious Count  Total Original Correct
Physician_Removed_Answer     Overall          10.662824              74                 